In [6]:
from pipeline import IndicatorSpec, build_feature_df
from data import fetch_binance_klines
from simulation.rules import Rule, RuleGroup, ALL, ANY
from simulation.simulator import Strategy, SimConfig, run_simulation, align_timeframes
from plot_simulation import plot_simulation

symbol = "BTCUSDT"
market = "spot"


In [7]:

# 1m features
df1 = fetch_binance_klines(symbol, "15m", start="2026-03-1", end=None, market=market)
ind_1m = [
    IndicatorSpec("rsi_divergence", tag="rsi14", config={"length": 14}),
    IndicatorSpec("stochastic", tag="st14", config={"k_length": 14, "d_length": 3, "smooth": 3}),
    IndicatorSpec("supertrend", tag="st10_3", config={"length": 10, "multiplier": 3}),
]
df1_feat, _, _ = build_feature_df(df1, tz="Asia/Karachi", ma_windows=[20], indicators=ind_1m)

# 5m features
df5 = fetch_binance_klines(symbol, "5m", start="2026-02-15", end=None, market=market)
ind_5m = [
    IndicatorSpec("rsi_divergence", tag="rsi14", config={"length": 14}),
    IndicatorSpec("stochastic", tag="st14", config={"k_length": 14, "d_length": 3, "smooth": 3}),
]
df5_feat, _, _ = build_feature_df(df5, tz="Asia/Karachi", ma_windows=[], indicators=ind_5m)

# Merge MTF into base (1m)
merged = align_timeframes(df1_feat, {"5m": df5_feat}, base_label="1m")

# Define rules:
open_1m = ALL(
    Rule("1m_rsi>55", lambda r: r["1m__rsi14__RSI"] > 55),
    Rule("1m_stoch_K>D", lambda r: r["1m__st14__K"] > r["1m__st14__D"]),
    Rule("1m_ST_buy_flip", lambda r: bool(r["1m__st10_3__BUY"]) if "1m__st10_3__BUY" in r else False),
)

open_5m = ALL(
    Rule("5m_rsi>55", lambda r: r["5m__rsi14__RSI"] > 55),
    Rule("5m_stoch_K>D", lambda r: r["5m__st14__K"] > r["5m__st14__D"]),
)

open_rules = ANY(open_1m, open_5m)  # OR between timeframes

close_rules = ANY(
    Rule("1m_rsi<45", lambda r: r["1m__rsi14__RSI"] < 45),
    Rule("1m_ST_sell_flip", lambda r: bool(r["1m__st10_3__SELL"]) if "1m__st10_3__SELL" in r else False),
)

strategy = Strategy(open_rules_long=open_rules, close_rules_long=close_rules)

res = run_simulation(
    df=merged,
    strategy=strategy,
    cfg=SimConfig(initial_cash=10_000, max_open_trades=2, fee_bps=5, slippage_bps=1),
    time_col="t",
    price_col="close",
)

res.stats, res.events.head(), res.equity_curve.tail()

KeyError: '5m__rsi14__RSI'

In [ ]:
plot_simulation(
    df_raw=df1,  # raw 1m df for candles
    symbol=symbol,
    interval="1m",
    market=market,
    trades=res.trades,
    ma_windows=[20],
    indicators=ind_1m,
    plot_cfg=PlotConfig(tz="Asia/Karachi", show_last="2d", height=1400, width=1900),
)